# Pricing model: season-level back-test on held-out players

**What is priced:** one big-five player-season (July to June). The insurer pays if the player has an injury that keeps the player out longer than the deferment period (60 days by default), and separately if an injury ends their career.

**How the three models combine:**

| piece | model | what it gives |
|---|---|---|
| λ | negative binomial GLM ([frequency.py](../model/frequency.py)) | expected injuries in the season, dispersion α |
| q(D) | tiered gamma GLM ([severity.py](../model/severity.py)) with a league term, fit on big-five injuries, averaged over the injury mix | P(an injury lasts more than D days) |
| c | logistic GLM ([career.py](../model/career.py)), fit on injuries in all leagues, averaged over the injury mix | P(an injury ends the career) |

The type of a future injury is unknown, so q and c are averages over the injury mix: q(D) = Σ_k w_k P(days > D | cell k, player), c = Σ_k w_k P(career-ending | cell k, player), where w_k is the share of injuries in each body region × injury type cell. The severity GLM and the mix are fit on the insured population: injuries that start in a big-five player-season from 2015/16, the same injuries the frequency model counts. Then, with injury counts negative binomial (mean λ, dispersion α) and each injury independently "covered" with probability q:

```
P(≥1 injury)              = 1 − (1 + αλ)^(−1/α)
P(≥1 injury > D days)     = 1 − (1 + αλq)^(−1/α)
P(career-ending injury)   = 1 − (1 + αλc)^(−1/α)
```

**The test:** every model is fit on training players only. The held-out set is the frequency model's test player-seasons (20% of players, the same players held out of the severity fit). For each one, λ, q(60) and c are computed from what is known at the season start (league, position, age, minutes in the last 12 months, career minutes, prior injuries, date) and compared with what actually happened that season.

**Headline (6,086 held-out player-seasons):**

- **P(≥1 injury) is well calibrated:** predicted 54.2% vs actual 52.8%, and every decile is within about 7 points (14% to 86% from lowest to highest decile).
- **P(≥1 injury > D days) is calibrated overall:** 11.9% predicted vs 12.1% actual at the 60-day deferment (×0.98), ×0.96 at 30 days, ×0.97 at 90 days, ×0.86 at 180 days. By decile the 60-day actual rises from 3.8% to 19.9% (predicted 4.2% to 22.5%).
- **Two changes got it there (section b):** fitting q on big-five injuries instead of all leagues (big-five injuries are shorter: 13% vs 21% over 60 days), and a league term in the severity GLM (leagues record injuries differently, so league moves λ and q in opposite directions). Before, with q fit on all leagues, the 60-day rate was over-predicted ×1.44 (17.4% vs 12.1%) and the top five deciles were flat in actual.
- **P(career-ending) is about 1.4× too high, with very few events:** 0.21% vs 0.15% (7 events in 4,814 player-seasons). The riskiest 20% of player-seasons contain 4 of the 7. The career GLM is still fit on all leagues: the big-five population has only 25 career-ending injuries.


In [1]:
import sys; sys.path.insert(0, "..")   # run from runs/: find the model/ and database/ packages
import numpy as np
import pandas as pd
from model import frequency as freq
from model import pricing
from model import severity as sev
from runs import backtest

bt = backtest.run()
nb, b = bt.attrs["nb"], bt.attrs["bundle"]
lab = bt[bt["career_labelled"]]
print(f"held out: {len(bt):,} player-seasons, {bt.pid.nunique():,} players, seasons "
      f"{bt.season.min()}/{bt.season.min() - 1999:02d} to {bt.season.max()}/{bt.season.max() - 1999:02d}; "
      f"frequency fit on {bt.attrs['n_train']:,} training player-seasons (NB alpha = {bt.attrs['alpha']:.3f})")
print(f"career-ending outcome known for {len(lab):,} player-seasons (seasons up to {lab.season.max()}/{lab.season.max() - 1999:02d}): "
      f"{int(lab.career_ending.sum())} career-ending")

held out: 6,086 player-seasons, 1,956 players, seasons 2015/16 to 2024/25; frequency fit on 24,154 training player-seasons (NB alpha = 0.218)
career-ending outcome known for 4,814 player-seasons (seasons up to 2022/23): 7 career-ending


## Overall: predicted vs actual

Each row is a season probability. `predicted` is the mean model probability over the held-out player-seasons; `actual` is the share that had the event. `events` is the number of player-seasons with the event.

- *≥1 injury*: any injury recorded in the season (the injuries the frequency model counts).
- *≥1 injury > D days*: an injury starting in the season with more than D days missed. Injuries still open at the data date count with the days so far; every season here ended more than a year before the data date, so open injuries are already past 180 days.
- *career-ending*: the season contains the player's career-ending injury (career.py's label: no later appearance, no return that season). Only seasons whose follow-up is complete are used.

In [2]:
rows = [("≥1 injury", "p_any", "any_injury", bt)]
rows += [(f"≥1 injury > {d} days", f"p_covered_{d}", f"over_{d}", bt) for d in backtest.DEFERMENTS]
rows += [("career-ending injury", "p_career", "career_ending", lab)]
overall = pd.DataFrame([{"probability": name, "n": len(d), "events": int(d[a].sum()),
                         "predicted": d[p].mean(), "actual": d[a].mean(), "ratio": d[p].mean() / d[a].mean()}
                        for name, p, a, d in rows]).set_index("probability")
overall.round(4)

,n,events,predicted,actual,ratio
probability,,,,,
≥1 injury,6086,3212,0.5424,0.5278,1.0277
≥1 injury > 30 days,6086,1538,0.2433,0.2527,0.9626
≥1 injury > 60 days,6086,738,0.1194,0.1213,0.9843
≥1 injury > 90 days,6086,441,0.0704,0.0725,0.9709
≥1 injury > 180 days,6086,168,0.0237,0.0276,0.8597
career-ending injury,4814,7,0.0021,0.0015,1.4445


## (a) P(≥1 injury) by predicted decile

Player-seasons are sorted by predicted probability and cut into ten equal groups (1 = lowest risk). A well-calibrated model has `predicted` ≈ `actual` in every row; the spread from decile 1 to 10 shows how well the model separates low- and high-risk players.

In [3]:
backtest.by_group(bt, "p_any", "any_injury").round(3)

,n,predicted,actual
decile,,,
1,609,0.213,0.141
2,609,0.312,0.230
3,608,0.369,0.336
4,609,0.434,0.430
5,608,0.510,0.543
6,609,0.583,0.606
7,608,0.648,0.628
8,609,0.715,0.747
9,608,0.778,0.757


## (b) P(≥1 injury > 60 days) by predicted decile

This is the main insured event (60-day deferment).

In [4]:
backtest.by_group(bt, "p_covered_60", "over_60").round(3)

,n,predicted,actual
decile,,,
1,609,0.042,0.038
2,609,0.063,0.056
3,608,0.075,0.094
4,609,0.087,0.112
5,608,0.103,0.130
6,609,0.121,0.130
7,608,0.139,0.128
8,609,0.158,0.153
9,608,0.181,0.174


**What changed and why.** q was first fit on injuries in every league, which over-predicted this table ×1.44. Two things were wrong:

1. *Population.* Big-five players' recorded injuries are shorter (Transfermarkt logs more minor knocks for them): 13% exceed 60 days vs 21% over all leagues. The severity GLM and the injury mix are now fit on the insured injuries only (`pricing.insured_injuries`: injuries starting in a big-five player-season).
2. *League.* Leagues record injuries differently: the Premier League has the lowest recorded rate but the longest injuries; Serie A and the Bundesliga record more and shorter ones. So league raises λ and lowers q, and without a league term in the severity GLM the Premier League's covered rate was under-predicted by a third and Serie A's over-predicted. With it, expected days are ×0.69 in Serie A, ×0.74 in the Bundesliga, ×0.84 in LaLiga and ×0.88 in Ligue 1 relative to the Premier League (table below).

Some count–severity dependence remains: players with a high λ have shorter injuries than q says (bottom table), but the season-level deciles above are now roughly monotone and within about 3 points.

In [5]:
inj = freq.covered_injuries().merge(bt[["pid", "season", "lam", "q_60"]], on=["pid", "season"])
all_share = (sev.completed(sev.build_dataset()).days_missed > 60).mean()
print(f"share of injuries > 60 days: all leagues {all_share:.3f}; held-out big-five injuries "
      f"{(inj.days_missed > 60).mean():.3f} (mean model q(60) for those players {inj.q_60.mean():.3f})")
lg = sev.coefficients(b["tiered"]).filter(like="league", axis=0)
display(lg.assign(days_ratio=np.exp(lg["coef"]).round(3)))
display(bt.groupby("league", observed=True).agg(n=("lam", "size"), mean_lambda=("lam", "mean"),
        predicted_60=("p_covered_60", "mean"), actual_60=("over_60", "mean")).round(3))
inj["lambda_quintile"] = pd.qcut(inj["lam"].rank(method="first"), 5, labels=range(1, 6))
inj.groupby("lambda_quintile", observed=True).agg(injuries=("lam", "size"), mean_lambda=("lam", "mean"),
    model_q60=("q_60", "mean"), actual_share_over_60=("days_missed", lambda x: (x > 60).mean())).round(3)

share of injuries > 60 days: all leagues 0.213; held-out big-five injuries 0.128 (mean model q(60) for those players 0.127)


,coef,p_value,days_ratio
variable,,,
league = ES1,-0.1721,0.0,0.842
league = IT1,-0.3762,0.0,0.686
league = L1,-0.2983,0.0,0.742
league = FR1,-0.1277,0.0,0.880


,n,mean_lambda,predicted_60,actual_60
league,,,,
GB1,1243,0.969,0.144,0.171
ES1,1189,0.814,0.104,0.102
IT1,1451,1.254,0.116,0.105
L1,1023,1.551,0.146,0.155
FR1,1180,0.641,0.090,0.078


,injuries,mean_lambda,model_q60,actual_share_over_60
lambda_quintile,,,,
1,1246,0.529,0.160,0.190
2,1246,0.956,0.140,0.135
3,1245,1.367,0.126,0.139
4,1246,1.813,0.112,0.096
5,1246,2.686,0.094,0.078


## (c) P(career-ending injury) by predicted quintile

Career-ending injuries are rare in big-five seasons (most players drop to lower leagues before they retire), so the events are few: read the overall rate and the capture number, not the individual quintiles. `capture` is the share of the actual career endings that fall in the riskiest 20% of player-seasons (20% would be no better than chance).

In [6]:
display(backtest.by_group(lab, "p_career", "career_ending", bins=5).round(4))
print(f"riskiest 20% of player-seasons contain {backtest.capture(lab, 'p_career', 'career_ending'):.0%} "
      f"of the {int(lab.career_ending.sum())} career endings")

,n,predicted,actual
quintile,,,
1,963,0.0003,0.0010
2,963,0.0006,0.0000
3,962,0.0012,0.0021
4,963,0.0023,0.0000
5,963,0.0061,0.0042


riskiest 20% of player-seasons contain 57% of the 7 career endings


## Frequency coefficients

From the NB GLM fit on training player-seasons. `rate_ratio` = exp(coef) multiplies the expected number of injuries: 1.30 means 30% more injuries per season, holding the rest fixed. The intercept is the rate for a 26-year-old Premier League midfielder with no injury history and no games last season; league terms partly reflect recording practice.

In [7]:
freq.coefficients(nb)

,coef,rate_ratio,p_value
variable,,,
Intercept,-1.4036,0.2457,0.0000
league = ES1,-0.0239,0.9764,0.3156
league = IT1,0.2895,1.3358,0.0000
league = L1,0.3094,1.3626,0.0000
league = FR1,-0.1218,0.8853,0.0000
position_group = Goalkeeper,-0.4610,0.6307,0.0000
position_group = Defender,0.0447,1.0457,0.0132
position_group = Attack,0.0878,1.0918,0.0000
age_c,-0.0071,0.9929,0.0014


## Deferment sensitivity for a typical player

A median held-out player-season (age, minutes last season, prior injuries, career minutes at the median; most common league and position). Longer deferments cut the chance of a claim sharply; the career-ending probability doesn't depend on the deferment.

In [8]:
med = bt[["age", "minutes_last_season", "prior_injuries", "career_games"]].median()
league, position = bt.league.astype(str).mode()[0], bt.position_group.astype(str).mode()[0]
lam = float(freq.expected_injuries(nb, freq.player_row(med.age, position, league, med.minutes_last_season,
                                                        typical=nb.typical))[0])
p = pricing.injury_probs(b, backtest.DEFERMENTS, age=med.age, position=position, league=league, prior_injuries=med.prior_injuries,
                         minutes_last_12_months=med.minutes_last_season, career_minutes=med.career_games * 90,
                         date="2024-07-01")
season = pricing.combine(lam, 1.0, p.career_ending, nb.alpha)
print(f"age {med.age:.0f}, {position}, {league}, {med.minutes_last_season:,.0f} minutes last season, "
      f"{med.prior_injuries:.0f} prior injuries: lambda = {lam:.2f}, P(>=1 injury) = {season.p_any_injury:.3f}, "
      f"c = {p.career_ending:.4f}, P(career-ending) = {season.p_career_ending:.4f}")
pd.DataFrame([{"deferment_days": d, "q": p[f"q_{d}"], "mean_days_if_covered": p[f"mean_days_over_{d}"],
               "p_covered_injury": pricing.combine(lam, p[f"q_{d}"], p.career_ending, nb.alpha)["p_covered_injury"]}
              for d in backtest.DEFERMENTS]).set_index("deferment_days").round(4)

age 25, Defender, IT1, 1,898 minutes last season, 2 prior injuries: lambda = 1.04, P(>=1 injury) = 0.609, c = 0.0020, P(career-ending) = 0.0021


,q,mean_days_if_covered,p_covered_injury
deferment_days,,,
30,0.2639,77.0951,0.2345
60,0.1120,125.0345,0.1090
90,0.0621,167.0442,0.0623
180,0.0187,263.7896,0.0193


## Caveats

- **Big-five only.** λ is fit on big-five first-division seasons, where injury recording is dense. It shouldn't be applied to other leagues as is.
- **Populations.** λ and q come from big-five seasons from 2015/16; c still comes from injuries in every league (too few career-ending injuries in the big five), and over-predicts slightly here.
- **Independence.** The thinning formula assumes the number of injuries, their type and their length are independent, and that injuries in a season are independent of each other. Frequent-injury players actually have shorter injuries on average; the league term in q removes the part of this due to league, the rest is small at season level.
- **NB thinning** is an approximation: it treats the gamma mixing in the NB as independent of which injuries are covered.
- **Unknown at quote time** and set to quote() defaults: re-injury within 60 days and same body part injured before (both 0). Position "Unknown" is treated as Midfield.
- **Part seasons.** A player who joins or leaves the big five mid-season counts as one full season.
- **Few career-ending events.** The career-ending check rests on a handful of events and is only indicative.